# Rupert Sim: Colab run (v2, saves to Google Drive + renders a video)
Free. Before running: **Runtime > Change runtime type > T4 GPU > Save**, then **Runtime > Run all**.
Colab will ask permission to use your Google Drive in step 2: click **Connect to Google Drive** and **Allow**. That is how the checkpoints (Rupert's learned brain) survive after Colab closes.
Training uses the tuned Playground setting, so it can take 30-60+ minutes on a T4.

In [ ]:
# 1. Check we really have a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Connect Google Drive (results are saved in My Drive > rupert > colab_run2)
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/rupert/colab_run2'

In [ ]:
# 3. Download the Rupert Sim code and install the tested versions
!rm -rf rupert-learning
!git clone https://github.com/kempyyyy4/rupert-learning.git
%cd rupert-learning
!pip install -q "jax[cuda12]==0.9.2" mujoco playground

In [ ]:
# 4. Train. The first line printed says how many steps this run will do. Watch 'reward' climb.
!python -m rupert.train --task HumanoidRun --out {OUT}
!cat {OUT}/config.json | grep -E "num_timesteps|num_envs|num_evals" 

In [ ]:
# 5. Learning curve
import json, matplotlib.pyplot as plt
log = json.load(open(OUT + '/progress.json'))
plt.plot([l['step'] for l in log], [l['reward'] for l in log], marker='o')
plt.xlabel('training steps'); plt.ylabel('reward (higher = better)'); plt.title('Rupert learning curve')
plt.show()

In [ ]:
# 6. Film Rupert with the final checkpoint and show the video
import os
os.environ['MUJOCO_GL'] = 'egl'
!python -m rupert.render --run {OUT} --task HumanoidRun
import glob, mediapy
mediapy.show_video(mediapy.read_video(sorted(glob.glob(OUT + '/*.mp4'))[-1]), fps=30)

## Done
Tell Claude: the last reward number, the `num_timesteps` line from step 4, and what the video looks like. Everything is in Google Drive > rupert > colab_run2.